In [1]:
import requests
from prebuilt_tools import get_tool_by_id

def _prebuilt_placeholder(tool_data):
    return get_tool_by_id(tool_data.get("id"))

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {"message": "Custom function placeholder", "tool": tool_data.get("name")}
    return _run

In [2]:
from langchain_core.tools import StructuredTool
from pydantic import create_model
import requests
from typing import Dict, Any, List

def _custom_api_placeholder(tool_def: Dict[str, Any]) -> List:
    """
    Convert custom_api tool definition to LangChain tool
    
    Returns:  List containing one LangChain StructuredTool
    """
    name = tool_def["name"]
    description = tool_def["description"]
    api_url = tool_def["api_url"]
    api_request_type = tool_def["api_request_type"]
    custom_message = tool_def.get("custom_message", "")
    input_schema = tool_def["input_schema"]
    
    # Build Pydantic model from input_schema
    fields = {}
    properties = input_schema.get("properties", {})
    
    for field_name, field_spec in properties.items():
        field_type_map = {
            "string": str,
            "number": float,
            "integer":  int,
            "boolean": bool
        }
        field_type = field_type_map.get(field_spec.get("type"), str)
        fields[field_name] = (field_type, None)
    
    # Fallback if no properties
    if not fields: 
        fields = {"_placeholder": (str, None)}
    
    InputModel = create_model(f"{name}_Input", **fields)
    
    # API call function
    def execute_api_call(**kwargs) -> Dict[str, Any]:
        try:
            # Remove placeholder if exists
            kwargs.pop("_placeholder", None)
            
            if api_request_type.upper() == "GET":
                resp = requests.get(api_url, params=kwargs, timeout=10)
            else:  # POST
                resp = requests.post(api_url, json=kwargs, timeout=10)
            
            return {
                "status_code": resp.status_code,
                "data": resp.json() if resp.content else {},
                "custom_message": custom_message
            }
        except Exception as e:
            return {
                "status_code": 500,
                "data": {"error":  str(e)},
                "custom_message": custom_message
            }
    
    # Create LangChain tool
    tool = StructuredTool.from_function(
        func=execute_api_call,
        name=name,
        description=description,
        args_schema=InputModel
    )
    
    return [tool]

In [3]:
from typing import List
from langchain_core.tools import StructuredTool
from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[StructuredTool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[StructuredTool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")
            continue

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Create tools based on type ---
        if tool_type == "prebuilt":
            tool = _prebuilt_placeholder(tool_data)
            print(tool)
            langchain_tools.append(tool)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_api":
            # _custom_api_placeholder returns a list of tools
            tools = _custom_api_placeholder(tool_data)
            langchain_tools.extend(tools)

    return langchain_tools

In [4]:
tool_ids = ["tool_517087cd-4f45-4dfb-835d-ec908086baa4","tool_fe925649-e6ec-4002-8d7f-7374d07ffa7d","tool_54da582b-e8e6-4f98-ab19-bfaf0a5965b6"]

tools = build_langchain_tools(tool_ids)

name='email_validator' description='Validate whether an email address is valid and non-disposable' args_schema=<class 'langchain_core.utils.pydantic.email_validator'> func=<function validate_email at 0x00000159944F6D40>


In [5]:
tools[0].invoke({"email": "rahul@1231231./as1@com"})

{'valid': False, 'reason': 'Email format is not correct'}

In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()


llm = ChatOpenAI(temperature=0)

agent = create_agent(model=llm, tools=tools)

response=agent.invoke({"messages": [("user", "rahul@1231231/com")]})
response

{'messages': [HumanMessage(content='rahul@1231231/com', additional_kwargs={}, response_metadata={}, id='3c0d77d5-2b09-4922-afd7-a60245d15adb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 122, 'total_tokens': 142, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CwtXFRsRpLTv4bxOgW6UZdOTrlFuP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bae1a-9de0-7552-b44a-e8b0d221f525-0', tool_calls=[{'name': 'email_validator', 'args': {'email': 'rahul@1231231/com'}, 'id': 'call_JKImVy5NGMrsbwzXqH7XSVhw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens

In [ ]:
response['messages'][-1].content